# 02 · 실제 모델 입력 데이터셋 EDA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lab-axis/Wind-Power-Forecasting/blob/main/notebooks/02_model_dataset_eda.ipynb)

전처리된 파일 전체와 모델이 실제로 사용할 수 있는 윈도우는 다릅니다. 이 노트북은 학습·검증·개발 평가 분할과 24시간 입력의 유효 표본을 확인합니다. 기상과 발전량의 관계는 관측상 연관이며 인과 효과나 개별 터빈 파워 커브가 아닙니다.

In [ ]:
from pathlib import Path
import os, sys

# Google Colab 자동 감지 및 환경 설정
if 'google.colab' in sys.modules or (os.path.exists('/content') and not (Path.cwd() / "configs/base_config.yaml").exists()):
    repo_dir = Path('/content/Wind-Power-Forecasting')
    if not repo_dir.exists():
        print("Google Colab 환경 감지: Wind-Power-Forecasting 저장소를 클론합니다...")
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/lab-axis/Wind-Power-Forecasting.git", str(repo_dir)], check=True)
        print("필수 의존 패키지를 설치합니다 (colab/requirements-colab.txt)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo_dir / "colab/requirements-colab.txt")], check=True)
    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))
    try:
        get_ipython().run_line_magic('cd', str(repo_dir))
    except Exception:
        pass

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/base_config.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
runtime = ROOT / ".experiment_archive/python_runtime"
if runtime.exists(): sys.path.insert(0, str(runtime))
from IPython.display import display
from src.workflows import research as wf
%matplotlib inline

In [ ]:
wf.split_quality()

## 1. 시계열·분포·영발전과 관측소 풍속

바람 그림은 보간하지 않은 관측 기상만 사용합니다. 그림을 보고 2025년을 다시 튜닝하는 절차는 포함하지 않습니다.

In [ ]:
wf.plot_dataset();

## 2. 공통 평가 표본

발행 시각 t의 입력은 [t−23h, t], 타깃은 t+h입니다. 배열의 exclusive index i는 입력 [i−24:i], 타깃 i+h−1에 대응합니다. 결측이 있으면 행을 삭제해 시간을 당기지 않고 해당 윈도우를 제외합니다. 분할 경계마다 윈도우를 새로 시작하므로 검증·평가의 첫 24시간을 과거 구간에서 보충하지 않습니다. 모든 모델을 같은 시각에 비교하기 위해 비기상 모델에도 이 공통 표본 규칙을 적용합니다.

In [ ]:
wf.window_counts()

## 3. 실제 한 개 입력과 타깃

아래 발전량 입력은 용량으로 나누고 기상 6채널은 학습 구간의 평균·표준편차로 변환했습니다. EMFN Dataset은 발전량을 MWh로 반환한 뒤 `EMFN.forward()`에서 /21을 수행하고, 일반 신경망 Dataset은 /21을 미리 수행합니다. 같은 물리 단위를 두 번 나누지 않습니다. 24h LightGBM은 기존 규약대로 MWh 발전량을 사용합니다.

In [ ]:
X, target = wf.input_example(split="validation", horizon=6)
display(X)
display(target)

## 4. 코드 추적 경로

`prepare_splits()` → train-only weather scaler → `valid_window_indices()` → `WindTimeSeriesDataset` / `EMFNMultiChannelDataset` / `flattened_windows()`. 입력 특징은 명시적인 7채널이며 `is_zero`, 원본 단위, 타깃 시각 풍속 등 정답·감사 열은 입력에 들어가지 않습니다.

**다음:** [03 · 베이스라인 학습과 평가](03_baseline_training_and_metrics.ipynb).